# Deepfake Hybrid Detection — Colab Training Notebook

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

### Dataset structures expected

**FaceForensics++ (FFPP)**
```
ffpp-dataset/
├── original_sequences/youtube/c23/videos/   ← real videos (.mp4)
└── manipulated_sequences/
    ├── Deepfakes/c23/videos/                ← fake videos (.mp4)
    └── Face2Face/c23/videos/               ← fake videos (.mp4)
```

**Celeb-DF v2 (CDF)**
```
celeb-df/
├── Celeb-real/          ← real celebrity videos (.mp4)
├── YouTube-real/        ← real YouTube videos (.mp4)
└── Celeb-synthesis/     ← fake synthesis videos (.mp4)
```

### Pipeline steps
1. Mount Google Drive & clone project code
2. Install dependencies
3. Copy datasets from Drive to local `/content/` (faster I/O)
4. Update `config.yaml` paths
5. Extract frames from videos (FFPP + CDF)
6. Build train/val/test splits (FFPP + CDF)
7. Compute FFT cache (FFPP + CDF)
8. Train models (on FFPP and/or CDF)
9. Evaluate & save outputs back to Drive
10. Save everything back to Drive

## Step 1 — Mount Drive & clone project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# ---------------------------------------------------------------------------
# OPTION A: your code is on GitHub — replace with your actual repo URL
# ---------------------------------------------------------------------------
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/skripsi

# ---------------------------------------------------------------------------
# OPTION B: your code is already in Google Drive
#   Change the path below to where your project folder lives in Drive
# ---------------------------------------------------------------------------
CODE_IN_DRIVE = "/content/drive/MyDrive/skripsi"   # <-- EDIT THIS
!cp -r "{CODE_IN_DRIVE}" /content/skripsi

os.chdir("/content/skripsi/deepfake_hybrid")
!pwd && ls

/content/skripsi/deepfake_hybrid
 colab_train.ipynb
 command.txt
 config.CDF.sampled.yaml
 config.FFPP.sampled.yaml
 config.yaml
 dataset
'Design Eksperimental Skripsi Giovanny - Kuantitatif.docx'
 outputs
 README.md
 requirements.txt
 scripts
 src


## Step 2 — Install dependencies

In [ ]:
!pip install -q timm scikit-learn tqdm opencv-python-headless
# ffmpeg is needed so OpenCV can decode H.264/H.265 .mp4 files
!apt-get install -qq ffmpeg libavcodec-extra
# torch + torchvision are pre-installed on Colab GPU runtimes
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Selecting previously unselected package libaribb24-0:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../libaribb24-0_1.0.3-2_amd64.deb ...
Unpacking libaribb24-0:amd64 (1.0.3-2) ...
Selecting previously unselected package libopencore-amrnb0:amd64.
Preparing to unpack .../libopencore-amrnb0_0.1.5-1_amd64.deb ...
Unpacking libopencore-amrnb0:amd64 (0.1.5-1) ...
Selecting previously unselected package libopencore-amrwb0:amd64.
Preparing to unpack .../libopencore-amrwb0_0.1.5-1_amd64.deb ...
Unpacking libopencore-amrwb0:amd64 (0.1.5-1) ...
Selecting previously unselected package libvo-amrwbenc0:amd64.
Preparing to unpack .../libvo-amrwbenc0_0.1.3-2_amd64.deb ...
Unpacking libvo-amrwbenc0:amd64 (0.1.3-2) ...
dpkg: libavcodec58:amd64: dependency problems, but removing anyway as you requested:
 libchromaprint1:amd64 depends on libavcodec58 (>= 7:4.4).
 libavformat58:amd64 depends on libavcodec58 (= 7:4.4.2-0ubuntu0.22.04.1).
 libavfilter7:a

## Step 3 — Copy / unzip datasets to local storage

Reading directly from Drive during training is very slow.
Both datasets are extracted to `/content/` first (fast local SSD).

**FFPP** — expected as a folder in Drive:
```
MyDrive/ffpp/
├── original_sequences/youtube/c23/videos/
└── manipulated_sequences/Deepfakes|Face2Face/c23/videos/
```

**Celeb-DF v2 (CDF)** — stored as a zip in Drive:
```
MyDrive/skripsi/deepfake_hybrid/dataset/celeb_df/Celeb-DF-v2.zip
```
The zip is extracted directly to `/content/celeb_df/` without copying the zip first.
After extraction the expected layout is:
```
/content/celeb_df/
├── Celeb-real/
├── YouTube-real/
└── Celeb-synthesis/
```

In [ ]:
import os

# ── FFPP ────────────────────────────────────────────────────────────────────
FFPP_IN_DRIVE = "/content/drive/MyDrive/ffpp"   # <-- EDIT THIS if path differs

real_path     = os.path.join(FFPP_IN_DRIVE, "original_sequences", "youtube", "c23", "videos")
fake_df_path  = os.path.join(FFPP_IN_DRIVE, "manipulated_sequences", "Deepfakes", "c23", "videos")
fake_f2f_path = os.path.join(FFPP_IN_DRIVE, "manipulated_sequences", "Face2Face", "c23", "videos")

print("=== FFPP verification ===")
for label, path in [("real (original)", real_path), ("fake (Deepfakes)", fake_df_path), ("fake (Face2Face)", fake_f2f_path)]:
    exists = os.path.isdir(path)
    count  = len([f for f in os.listdir(path) if f.endswith(".mp4")]) if exists else 0
    print(f"{'OK' if exists else 'MISSING'} [{label}] {path} — {count} .mp4 files")

# ── CDF (zip) ────────────────────────────────────────────────────────────────
CDF_ZIP = "/content/drive/MyDrive/skripsi/deepfake_hybrid/dataset/celeb_df/Celeb-DF-v2.zip"  # <-- EDIT THIS if path differs

print("\n=== CDF verification ===")
if os.path.isfile(CDF_ZIP):
    size_gb = os.path.getsize(CDF_ZIP) / 1_073_741_824
    print(f"OK  [zip found] {CDF_ZIP} — {size_gb:.2f} GB")
else:
    print(f"MISSING [zip not found] {CDF_ZIP}")
    print("    Check that the path above matches where Celeb-DF-v2.zip lives in your Drive.")

In [ ]:
import os

# ── FFPP: copy folder from Drive ─────────────────────────────────────────────
print("Copying FFPP from Drive (this may take a few minutes)...")
!cp -r "{FFPP_IN_DRIVE}" /content/ffpp
print("Done.")
!ls /content/ffpp

# ── CDF: unzip directly from Drive to local SSD ──────────────────────────────
# Unzipping straight from Drive avoids copying the large zip to local disk first.
# The -q flag suppresses per-file output; remove it to see progress.
print("\nUnzipping Celeb-DF-v2.zip from Drive to /content/celeb_df/ ...")
os.makedirs("/content/celeb_df", exist_ok=True)
!unzip -q "{CDF_ZIP}" -d /content/celeb_df
print("Done.")

# The zip may extract into a subdirectory (e.g. Celeb-DF-v2/).
# Flatten it so /content/celeb_df/ contains Celeb-real/, YouTube-real/, Celeb-synthesis/ directly.
import shutil
from pathlib import Path

celeb_df_root = Path("/content/celeb_df")
subdirs = [d for d in celeb_df_root.iterdir() if d.is_dir()]
expected = {"Celeb-real", "YouTube-real", "Celeb-synthesis"}

if subdirs and not expected.intersection({d.name for d in subdirs}):
    # Likely extracted into one wrapper subfolder — move contents up
    wrapper = subdirs[0]
    print(f"Detected wrapper folder '{wrapper.name}', moving contents up...")
    for item in wrapper.iterdir():
        shutil.move(str(item), str(celeb_df_root / item.name))
    wrapper.rmdir()

print("\nTop-level contents of /content/celeb_df/:")
!ls /content/celeb_df

# Sanity check
print("\nSample files:")
for subdir in ["Celeb-real", "YouTube-real", "Celeb-synthesis"]:
    p = celeb_df_root / subdir
    if p.exists():
        mp4s = list(p.glob("*.mp4"))
        print(f"  {subdir}/  — {len(mp4s)} .mp4 files  (first: {mp4s[0].name if mp4s else 'none'})")
    else:
        print(f"  {subdir}/  — MISSING")

## Step 4 — Update config.yaml paths

In [ ]:
import yaml
from pathlib import Path

CONFIG_PATH = "/content/skripsi/deepfake_hybrid/config.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# ── FFPP ────────────────────────────────────────────────────────────────────
cfg['ffpp_root'] = "/content/ffpp"
cfg['datasets']['ffpp']['root'] = "/content/ffpp"
cfg['datasets']['ffpp']['real_keywords'] = ["original", "real", "pristine", "actors", "youtube"]
cfg['datasets']['ffpp']['fake_keywords'] = ["fake", "manipulated", "deepfakes", "faceswap",
                                             "neuraltextures", "deepfakedetection", "faceshifter",
                                             "face2face"]

# ── CDF ─────────────────────────────────────────────────────────────────────
# Celeb-DF v2 folder names: Celeb-real/, YouTube-real/ (real) and Celeb-synthesis/ (fake)
cfg['celebdf_root'] = "/content/celeb_df"
cfg['datasets']['cdf']['root'] = "/content/celeb_df"
cfg['datasets']['cdf']['real_keywords'] = ["real", "authentic", "youtube"]
cfg['datasets']['cdf']['fake_keywords'] = ["fake", "synthesis", "deepfake"]

# ── Shared settings ──────────────────────────────────────────────────────────
cfg['output_root'] = "/content/outputs"
cfg['epochs'] = 20          # change to 30-50 for a full run
cfg['n_seeds'] = 1          # increase to 3 for statistical validity
cfg['batch_size'] = 16      # reduce to 8 if you get OOM errors
cfg['num_workers'] = 2      # Colab works well with 2

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f)

print("Config updated:")
with open(CONFIG_PATH) as f:
    print(f.read())

## Step 5 — Extract frames from videos

`extract_frames.py` recursively scans the dataset root for `.mp4` files and infers labels from folder names.

**FFPP label rules:**
- `original_sequences/youtube/...` → **real** (label 0)
- `manipulated_sequences/Deepfakes/...` → **fake** (label 1)
- `manipulated_sequences/Face2Face/...` → **fake** (label 1)

**CDF label rules:**
- `Celeb-real/` or `YouTube-real/` → **real** (label 0)
- `Celeb-synthesis/` → **fake** (label 1)

**If you get an empty manifest error**, run the debug cell below first to diagnose.

In [ ]:
import pandas as pd
from pathlib import Path

def extract_and_check(dataset_name, local_root):
    print(f"\n{'='*50}")
    print(f"Extracting frames: {dataset_name}")
    print(f"{'='*50}")
    !python /content/skripsi/deepfake_hybrid/scripts/extract_frames.py \
        --config "{CONFIG_PATH}" \
        --dataset "{dataset_name}" \
        --fps 5 \
        --max-frames 100

    manifest_path = Path(f"/content/outputs/manifests/{dataset_name}/manifest.csv")
    assert manifest_path.exists(), f"manifest.csv was not created for {dataset_name} — extract_frames.py may have crashed"

    manifest = pd.read_csv(manifest_path)
    if len(manifest) == 0:
        raise RuntimeError(
            f"manifest.csv is empty for {dataset_name} — no videos were successfully processed.\n"
            f"  Check that {local_root} contains .mp4 files and that OpenCV can open them."
        )

    real_n = manifest['label'].eq(0).sum()
    fake_n = manifest['label'].eq(1).sum()
    print(f"  Total: {len(manifest)} | Real: {real_n} | Fake: {fake_n}")
    if manifest['label'].isna().any():
        print(f"  WARNING: {manifest['label'].isna().sum()} videos had unknown labels and were skipped")
    if real_n == 0:
        raise RuntimeError(f"0 real videos found in {dataset_name} — check folder names match real_keywords in config")
    if fake_n == 0:
        raise RuntimeError(f"0 fake videos found in {dataset_name} — check folder names match fake_keywords in config")

extract_and_check("FFPP", "/content/ffpp")
extract_and_check("CDF",  "/content/celeb_df")

In [ ]:
# ── DEBUG (run this if Step 5 gives an empty manifest) ──────────────────────
import cv2, glob

for ds, root in [("FFPP", "/content/ffpp"), ("CDF", "/content/celeb_df")]:
    print(f"\n=== {ds}: .mp4 files under {root} ===")
    mp4s = glob.glob(f"{root}/**/*.mp4", recursive=True)
    if mp4s:
        for p in mp4s[:5]:
            print(f"  {p}")
        cap = cv2.VideoCapture(mp4s[0])
        print(f"  OpenCV can open {mp4s[0]}: {cap.isOpened()}")
        cap.release()
    else:
        print(f"  No .mp4 files found — copy in Step 3 may have failed")
# ─────────────────────────────────────────────────────────────────────────────

## Step 6 — Build train / val / test splits

In [ ]:
import pandas as pd
from pathlib import Path

for dataset in ["FFPP", "CDF"]:
    manifest_path = Path(f"/content/outputs/manifests/{dataset}/manifest.csv")
    if not manifest_path.exists():
        raise FileNotFoundError(f"manifest.csv not found for {dataset} — re-run Step 5 first")
    manifest = pd.read_csv(manifest_path)
    if len(manifest) == 0:
        raise RuntimeError(f"manifest.csv is empty for {dataset} — re-run Step 5 and fix the issue it reports")
    print(f"\n{dataset}: {len(manifest)} videos in manifest — building splits...")

    !python /content/skripsi/deepfake_hybrid/scripts/build_splits.py \
        --config "{CONFIG_PATH}" \
        --dataset "{dataset}" \
        --val-size 0.15 \
        --test-size 0.15 \
        --seed 42

    for split in ['train', 'val', 'test']:
        df = pd.read_csv(f"/content/outputs/manifests/{dataset}/{split}.csv")
        print(f"  {split}: {len(df)} videos | real={df['label'].eq(0).sum()} fake={df['label'].eq(1).sum()}")

## Step 7 — Compute FFT cache
Only needed for `freq`, `hybrid`, and `early_fusion` models. Skip if training `spatial` only.

In [ ]:
for dataset in ["FFPP", "CDF"]:
    print(f"\nComputing FFT cache for {dataset}...")
    !python /content/skripsi/deepfake_hybrid/scripts/compute_fft_cache.py \
        --config "{CONFIG_PATH}" \
        --dataset "{dataset}" \
        --num-workers 2
    print(f"  Done. Cache saved under /content/outputs/fft_cache/{dataset}/")

## Step 8 — Train

Each cell below trains one model type on one dataset.
Run all cells sequentially for the full 4-model × 2-dataset experiment,
or pick only the combinations you need.

In [ ]:
# --- Spatial-only (XceptionNet baseline) ---
for dataset in ["FFPP", "CDF"]:
    print(f"\n=== Training spatial on {dataset} ===")
    !python /content/skripsi/deepfake_hybrid/scripts/train.py \
        --config "{CONFIG_PATH}" \
        --dataset "{dataset}" \
        --model spatial \
        --seed 42 \
        --pretrained

In [ ]:
# --- Frequency-only (FreqCNN baseline) ---
for dataset in ["FFPP", "CDF"]:
    print(f"\n=== Training freq on {dataset} ===")
    !python /content/skripsi/deepfake_hybrid/scripts/train.py \
        --config "{CONFIG_PATH}" \
        --dataset "{dataset}" \
        --model freq \
        --seed 42

In [ ]:
# --- Hybrid two-branch (main model) ---
for dataset in ["FFPP", "CDF"]:
    print(f"\n=== Training hybrid on {dataset} ===")
    !python /content/skripsi/deepfake_hybrid/scripts/train.py \
        --config "{CONFIG_PATH}" \
        --dataset "{dataset}" \
        --model hybrid \
        --seed 42 \
        --pretrained

In [ ]:
# --- Early fusion (alternative fusion) ---
for dataset in ["FFPP", "CDF"]:
    print(f"\n=== Training early_fusion on {dataset} ===")
    !python /content/skripsi/deepfake_hybrid/scripts/train.py \
        --config "{CONFIG_PATH}" \
        --dataset "{dataset}" \
        --model early_fusion \
        --seed 42 \
        --pretrained

## Step 9 — Evaluate checkpoints

In [ ]:
from pathlib import Path

# Evaluate every checkpoint on the dataset it was trained on (in-dataset eval).
# The checkpoint naming convention is: {model}_{DATASET}_seed{seed}/best.pt
for dataset in ["FFPP", "CDF"]:
    for model_type in ['spatial', 'freq', 'hybrid', 'early_fusion']:
        ckpt = f"/content/outputs/runs/{model_type}_{dataset}_seed42/best.pt"
        if not Path(ckpt).exists():
            print(f"[SKIP] No checkpoint: {model_type} on {dataset}")
            continue
        print(f"\n=== Evaluating {model_type} trained on {dataset} ===")
        !python /content/skripsi/deepfake_hybrid/scripts/eval.py \
            --config "{CONFIG_PATH}" \
            --dataset "{dataset}" \
            --model "{model_type}" \
            --checkpoint "{ckpt}" \
            --seed 42 \
            --pretrained

In [ ]:
# Print a summary table of all results
import json
from pathlib import Path

results_dir = Path("/content/outputs/tables")
rows = []
for f in sorted(results_dir.glob("*.json")):
    data = json.loads(f.read_text())
    rows.append({"experiment": f.stem, **data})

if rows:
    import pandas as pd
    df = pd.DataFrame(rows).set_index("experiment")
    pd.set_option('display.float_format', '{:.4f}'.format)
    print(df.to_string())
else:
    print("No results yet — run Step 9 first.")

## Step 10 — Save everything back to Drive

**Important:** Colab resets when the session ends. Always run this cell before closing.

In [ ]:
import shutil, os
from pathlib import Path
from datetime import datetime

# ── Where to save in your Drive ────────────────────────────────────────────
# A timestamped subfolder so each run doesn't overwrite the previous one.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_OUT = f"/content/drive/MyDrive/skripsi_outputs/{timestamp}"
os.makedirs(DRIVE_OUT, exist_ok=True)

LOCAL_OUT = "/content/outputs"

# ── 1. Model checkpoints (most important — save these first) ────────────────
runs_src = Path(LOCAL_OUT) / "runs"
if runs_src.exists():
    shutil.copytree(runs_src, f"{DRIVE_OUT}/runs")
    print(f"✓ Checkpoints saved → {DRIVE_OUT}/runs/")
else:
    print("⚠ No runs/ directory found — training may not have completed")

# ── 2. Results tables (JSON / CSV metrics) ──────────────────────────────────
tables_src = Path(LOCAL_OUT) / "tables"
if tables_src.exists():
    shutil.copytree(tables_src, f"{DRIVE_OUT}/tables")
    print(f"✓ Results saved   → {DRIVE_OUT}/tables/")

# ── 3. Manifests / splits ────────────────────────────────────────────────────
manifests_src = Path(LOCAL_OUT) / "manifests"
if manifests_src.exists():
    shutil.copytree(manifests_src, f"{DRIVE_OUT}/manifests")
    print(f"✓ Manifests saved → {DRIVE_OUT}/manifests/")

# ── 4. Updated config.yaml ───────────────────────────────────────────────────
config_src = Path("/content/skripsi/deepfake_hybrid/config.yaml")
if config_src.exists():
    shutil.copy2(config_src, f"{DRIVE_OUT}/config.yaml")
    print(f"✓ config.yaml saved → {DRIVE_OUT}/config.yaml")

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\nAll outputs saved to Drive folder:\n  {DRIVE_OUT}")
print("\nContents:")
for item in sorted(Path(DRIVE_OUT).rglob("*")):
    if item.is_file():
        size_mb = item.stat().st_size / 1_048_576
        print(f"  {item.relative_to(DRIVE_OUT)}  ({size_mb:.1f} MB)")

✓ Checkpoints saved → /content/drive/MyDrive/skripsi_outputs/20260220_001723/runs/
✓ Results saved   → /content/drive/MyDrive/skripsi_outputs/20260220_001723/tables/
✓ Manifests saved → /content/drive/MyDrive/skripsi_outputs/20260220_001723/manifests/
✓ config.yaml saved → /content/drive/MyDrive/skripsi_outputs/20260220_001723/config.yaml

All outputs saved to Drive folder:
  /content/drive/MyDrive/skripsi_outputs/20260220_001723

Contents:
  config.yaml  (0.0 MB)
  manifests/FFPP/manifest.csv  (0.0 MB)
  manifests/FFPP/test.csv  (0.0 MB)
  manifests/FFPP/train.csv  (0.0 MB)
  manifests/FFPP/val.csv  (0.0 MB)
  runs/eval.log  (0.0 MB)
  runs/hybrid_FFPP_seed42/best.pt  (81.8 MB)
  runs/hybrid_FFPP_seed42/metrics.json  (0.0 MB)
  runs/hybrid_FFPP_seed42/train.log  (0.0 MB)
  tables/eval_hybrid_FFPP.json  (0.0 MB)


---
## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Reduce `batch_size` to `8` in Step 4 |
| `EmptyDataError: No columns to parse` in Step 6 | manifest.csv is empty — re-run Step 5; check the debug cell to see if `.mp4` files exist |
| `FileNotFoundError: manifest` in Step 6 | Re-run Step 5 (extract frames) first |
| Step 3 shows `MISSING` for FFPP real/fake paths | `FFPP_IN_DRIVE` must point to the folder that contains both `original_sequences/` and `manipulated_sequences/` |
| Step 3 shows `MISSING [zip not found]` for CDF | Update `CDF_ZIP` to the correct path of `Celeb-DF-v2.zip` in your Drive |
| Unzip fails with `End-of-central-directory` error | The zip in Drive is corrupted — re-upload it |
| After unzip, `Celeb-real/` etc. are MISSING | The zip extracted into a wrapper subfolder — the cell handles this automatically; if it fails, run `!ls /content/celeb_df` to see the actual layout |
| Debug cell shows `OpenCV can open ...: False` | Re-run Step 2 to install `ffmpeg`, then restart the runtime and re-run from Step 2 |
| Debug cell shows "No .mp4 files found" | Unzip or copy in Step 3 failed — re-run Step 3 |
| FFPP: all videos have `unknown label` warning | Verify subfolders are exactly `original_sequences`, `Deepfakes`, `Face2Face` (case-sensitive) |
| CDF: all videos have `unknown label` warning | Verify top-level subfolders are exactly `Celeb-real`, `YouTube-real`, `Celeb-synthesis` (case-sensitive) |
| Real count is 0, only fakes found (FFPP) | `original_sequences/youtube/c23/videos/` is missing or empty |
| Real count is 0, only fakes found (CDF) | `Celeb-real/` and `YouTube-real/` are missing or empty |
| Session disconnects mid-training | Outputs in `/content/` are lost — re-run Step 10 next session to restore from Drive |
| Slow unzip / copy | Normal — Drive I/O is ~30 MB/s. Let Step 3 finish before proceeding |